### Validate the generated DB

#### Imports & config

In [1]:
import sqlite3, os, shutil
from datetime import datetime

In [2]:
DB_PATH = "../backend/movies.db"

In [3]:
EXPECTED = {
    "users": 610,
    "movies": 9742,
    "user_recs_per_user": 20,
    "user_rec_rows": 610 * 20,
    "movies_with_similarity": 9742 - 5,
    "popular_rows": 100,
    "rating_count": 100836,
    "min_db_bytes": 1_000_000,
    "demo_users": [1, 57, 414, 610],
    "gap_movies": {129250, 155589, 156605, 169034, 171495},
}

TABLES = {"movies", "users", "popular_movies", "user_recommendations",
          "movie_similarity", "ratings_summary"}

INDEXES = {"idx_user_recommendations_user_id", "idx_movie_similarity_movie_id",
           "idx_movies_title", "idx_popular_movies_rank"}

SCHEMA = {
    "movies": {"movie_id", "title", "genres", "year", "imdb_id", "tmdb_id"},
    "users": {"user_id", "rating_count", "avg_rating"},
    "popular_movies": {"rank", "movie_id", "score", "rating_count", "avg_rating"},
    "user_recommendations": {"user_id", "rank", "movie_id", "score"},
    "movie_similarity": {"movie_id", "rank", "similar_movie_id", "score"},
    "ratings_summary": {"key", "value"},
}

#### Check registry + runner

Each check is `(name, fn)`; `fn(conn) -> (ok, detail)`. The runner opens the DB **read-only** (`mode=ro` — validation must never be able to mutate the artifact), runs everything, prints one line per check, and returns the overall verdict.

In [4]:
CHECKS = []

def check(name):
    """Register a validation check. fn(conn) -> (ok: bool, detail: str)."""
    def wrap(fn):
        CHECKS.append((name, fn))
        return fn
    return wrap

def scalar(conn, sql):
    return conn.execute(sql).fetchone()[0]

In [5]:
def run(db_path):
    """Run every check against db_path. Reports all results; returns overall verdict."""
    results = []

    # A1/A2: file-level pre-flight (before any SQL can run)
    exists = os.path.exists(db_path)
    size = os.path.getsize(db_path) if exists else 0
    results.append(("A1 file exists and >= 1 MB",
                    exists and size >= EXPECTED["min_db_bytes"],
                    f"{size/1e6:.2f} MB" if exists else "missing"))

    conn = None
    if exists:
        try:
            conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)  # READ-ONLY
            conn.execute("SELECT 1 FROM sqlite_master LIMIT 1")
            results.append(("A2 opens read-only as SQLite", True, ""))
        except sqlite3.Error as e:
            results.append(("A2 opens read-only as SQLite", False, str(e)))
            conn = None

    if conn is not None:
        for name, fn in CHECKS:
            try:
                ok, detail = fn(conn)
            except Exception as e:              # a crashing check is a failing check
                ok, detail = False, f"{type(e).__name__}: {e}"
            results.append((name, ok, detail))
        conn.close()

    failed = [name for name, ok, _ in results if not ok]
    for name, ok, detail in results:
        print(("PASS  " if ok else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))
    print(f"\n{len(results) - len(failed)}/{len(results)} checks passed")
    return not failed

#### A. Existence & structure

In [6]:
@check("A3 all six tables exist")
def a3(conn):
    have = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    missing = TABLES - have
    return (not missing, f"missing: {sorted(missing)}" if missing else "")

@check("A4 all four indexes exist")
def a4(conn):
    have = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='index'")}
    missing = INDEXES - have
    return (not missing, f"missing: {sorted(missing)}" if missing else "")

@check("A5 expected columns per table")
def a5(conn):
    bad = []
    for t, cols in SCHEMA.items():
        have = {r[1] for r in conn.execute(f'PRAGMA table_info("{t}")')}
        if have != cols:
            bad.append(f"{t}: missing {sorted(cols - have)}, extra {sorted(have - cols)}")
    return (not bad, "; ".join(bad))

#### B. Counts & coverage

In [7]:
@check("B1 movies count")
def b1(conn):
    n = scalar(conn, "SELECT COUNT(*) FROM movies")
    return (n == EXPECTED["movies"], f"{n}")

@check("B2 users count")
def b2(conn):
    n = scalar(conn, "SELECT COUNT(*) FROM users")
    return (n == EXPECTED["users"], f"{n}")

@check("B3 popular_movies count")
def b3(conn):
    n = scalar(conn, "SELECT COUNT(*) FROM popular_movies")
    return (n == EXPECTED["popular_rows"], f"{n}")

@check("B4 user_recommendations rows + full user coverage")
def b4(conn):
    rows = scalar(conn, "SELECT COUNT(*) FROM user_recommendations")
    users = scalar(conn, "SELECT COUNT(DISTINCT user_id) FROM user_recommendations")
    ok = rows == EXPECTED["user_rec_rows"] and users == EXPECTED["users"]
    return (ok, f"{rows} rows, {users} users")

@check("B5 movies with similarity rows (expected 5-gap)")
def b5(conn):
    n = scalar(conn, "SELECT COUNT(DISTINCT movie_id) FROM movie_similarity")
    gap = EXPECTED["movies"] - n
    return (n == EXPECTED["movies_with_similarity"], f"{n} (gap {gap})")

@check("B6 ratings_summary has 5 keys incl generated_at")
def b6(conn):
    keys = {r[0] for r in conn.execute("SELECT key FROM ratings_summary")}
    ok = len(keys) == 5 and "generated_at" in keys
    return (ok, ", ".join(sorted(keys)))

#### C. Referential integrity (Q17, re-asserted on the artifact)

The DDL declares FKs, but SQLite does **not enforce them** unless `PRAGMA foreign_keys=ON` was set at write time — these checks are the real enforcement.

In [8]:
ORPHAN_CHECKS = [
    ("C1 user_recommendations.movie_id -> movies",
     "SELECT COUNT(*) FROM user_recommendations t LEFT JOIN movies m ON t.movie_id = m.movie_id WHERE m.movie_id IS NULL"),
    ("C2 movie_similarity.movie_id -> movies",
     "SELECT COUNT(*) FROM movie_similarity t LEFT JOIN movies m ON t.movie_id = m.movie_id WHERE m.movie_id IS NULL"),
    ("C3 movie_similarity.similar_movie_id -> movies",
     "SELECT COUNT(*) FROM movie_similarity t LEFT JOIN movies m ON t.similar_movie_id = m.movie_id WHERE m.movie_id IS NULL"),
    ("C4 popular_movies.movie_id -> movies",
     "SELECT COUNT(*) FROM popular_movies t LEFT JOIN movies m ON t.movie_id = m.movie_id WHERE m.movie_id IS NULL"),
    ("C5 user_recommendations.user_id -> users",
     "SELECT COUNT(*) FROM user_recommendations t LEFT JOIN users u ON t.user_id = u.user_id WHERE u.user_id IS NULL"),
]

for _name, _sql in ORPHAN_CHECKS:
    def _orphan(conn, sql=_sql):
        n = scalar(conn, sql)
        return (n == 0, f"{n} orphans" if n else "")
    check(_name)(_orphan)

#### D. Ranking invariants

Rank is the authoritative order (build.md Stage 8) — validate *rank*, not score order.

In [9]:
@check("D1 no self-similarity")
def d1(conn):
    n = scalar(conn, "SELECT COUNT(*) FROM movie_similarity WHERE movie_id = similar_movie_id")
    return (n == 0, f"{n} rows" if n else "")

@check("D2 movie_similarity ranks contiguous 1..N per movie")
def d2(conn):
    n = scalar(conn, '''
        SELECT COUNT(*) FROM (
            SELECT movie_id FROM movie_similarity
            GROUP BY movie_id
            HAVING MIN("rank") != 1 OR MAX("rank") != COUNT(*) OR COUNT(DISTINCT "rank") != COUNT(*)
        )''')
    return (n == 0, f"{n} bad groups" if n else "")

@check("D3 user_recommendations: exactly 20, ranks contiguous, per user")
def d3(conn):
    n = scalar(conn, f'''
        SELECT COUNT(*) FROM (
            SELECT user_id FROM user_recommendations
            GROUP BY user_id
            HAVING COUNT(*) != {EXPECTED["user_recs_per_user"]}
                OR MIN("rank") != 1 OR MAX("rank") != COUNT(*)
                OR COUNT(DISTINCT "rank") != COUNT(*)
        )''')
    return (n == 0, f"{n} bad groups" if n else "")

@check("D4 popular_movies ranks are exactly 1..100")
def d4(conn):
    lo, hi, cnt, dst = conn.execute(
        'SELECT MIN("rank"), MAX("rank"), COUNT(*), COUNT(DISTINCT "rank") FROM popular_movies').fetchone()
    want = EXPECTED["popular_rows"]
    ok = (lo, hi, cnt, dst) == (1, want, want, want)
    return (ok, f"min {lo}, max {hi}, n {cnt}")

@check("D5 no duplicate targets within a list")
def d5(conn):
    a = scalar(conn, "SELECT COUNT(*) FROM (SELECT 1 FROM user_recommendations GROUP BY user_id, movie_id HAVING COUNT(*) > 1)")
    b = scalar(conn, "SELECT COUNT(*) FROM (SELECT 1 FROM movie_similarity GROUP BY movie_id, similar_movie_id HAVING COUNT(*) > 1)")
    return (a == 0 and b == 0, f"recs {a}, sims {b}" if a or b else "")

#### E. Value sanity

In [ ]:
@check("E1 no NULL scores anywhere")
def e1(conn):
    n = sum(scalar(conn, f"SELECT COUNT(*) FROM {t} WHERE score IS NULL")
            for t in ("popular_movies", "user_recommendations", "movie_similarity"))
    return (n == 0, f"{n} NULLs" if n else "")

@check("E2 popular_movies values plausible")
def e2(conn):
    n = scalar(conn, '''
        SELECT COUNT(*) FROM popular_movies
        WHERE score NOT BETWEEN 0.5 AND 5.0
           OR avg_rating NOT BETWEEN 0.5 AND 5.0
           OR rating_count < 1''')
    return (n == 0, f"{n} bad rows" if n else "")

@check("E3 users values plausible")
def e3(conn):
    n = scalar(conn, '''
        SELECT COUNT(*) FROM users
        WHERE rating_count < 20 OR avg_rating NOT BETWEEN 0.5 AND 5.0''')
    return (n == 0, f"{n} bad rows" if n else "")

@check("E4 movies sane (title/genres non-empty, year NULL or 1900-2030)")
def e4(conn):
    n = scalar(conn, '''
        SELECT COUNT(*) FROM movies
        WHERE title IS NULL OR TRIM(title) = \'\'
           OR genres IS NULL OR TRIM(genres) = \'\'
           OR (year IS NOT NULL AND year NOT BETWEEN 1900 AND 2030)''')
    return (n == 0, f"{n} bad rows" if n else "")

@check("E5 similarity scores within [0, 1] (float tolerance)")
def e5(conn):
    n = scalar(conn, "SELECT COUNT(*) FROM movie_similarity WHERE score < -1e-9 OR score > 1.000000001")
    return (n == 0, f"{n} out of range" if n else "")


@check("E6 imdb_id present and 7-digit zero-padded for all movies")
def e6(conn):
    n = scalar(conn, """
        SELECT COUNT(*) FROM movies
        WHERE imdb_id IS NULL
           OR LENGTH(imdb_id) != 7
           OR imdb_id NOT GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9]'""")
    return (n == 0, f"{n} bad rows" if n else "")


@check("E7 tmdb_id nullable with exactly the 8 known gaps")
def e7(conn):
    nulls = scalar(conn, "SELECT COUNT(*) FROM movies WHERE tmdb_id IS NULL")
    bad = scalar(conn, "SELECT COUNT(*) FROM movies WHERE tmdb_id IS NOT NULL AND tmdb_id < 1")
    return (nulls == 8 and bad == 0, f"{nulls} nulls, {bad} non-positive")

#### F. API-contract checks (phrased as the backend will query)

In [11]:
@check("F1 demo users (1, 57, 414, 610) each have exactly 20 recs")
def f1(conn):
    ids = ",".join(map(str, EXPECTED["demo_users"]))
    n = scalar(conn, f'''
        SELECT COUNT(*) FROM (
            SELECT user_id FROM user_recommendations
            WHERE user_id IN ({ids})
            GROUP BY user_id HAVING COUNT(*) = {EXPECTED["user_recs_per_user"]}
        )''')
    want = len(EXPECTED["demo_users"])
    return (n == want, f"{n}/{want} demo users ok")

@check("F2 unknown user 9999 -> no recs, popular fallback servable")
def f2(conn):
    unknown = scalar(conn, "SELECT COUNT(*) FROM user_recommendations WHERE user_id = 9999")
    pop = scalar(conn, "SELECT COUNT(*) FROM popular_movies")
    return (unknown == 0 and pop > 0, f"unknown rows {unknown}, popular {pop}")

@check("F3 title search servable (\'matrix\')")
def f3(conn):
    n = scalar(conn, "SELECT COUNT(*) FROM movies WHERE title LIKE \'%matrix%\'")
    return (n >= 1, f"{n} matches")

@check("F4 similar-movies path: hub full, known gaps empty")
def f4(conn):
    toy = scalar(conn, "SELECT COUNT(*) FROM movie_similarity WHERE movie_id = 1")
    gaps = {r[0] for r in conn.execute(
        "SELECT movie_id FROM movies WHERE movie_id NOT IN (SELECT DISTINCT movie_id FROM movie_similarity)")}
    ok = toy == 15 and gaps == EXPECTED["gap_movies"]
    return (ok, f"toy story {toy} rows; gaps {sorted(gaps)}")

#### G. Summary consistency (the DB describing itself truthfully)

In [12]:
@check("G1 summary movie_count matches movies table")
def g1(conn):
    s = scalar(conn, "SELECT CAST(value AS INTEGER) FROM ratings_summary WHERE key = \'movie_count\'")
    n = scalar(conn, "SELECT COUNT(*) FROM movies")
    return (s == n, f"summary {s} vs actual {n}")

@check("G2 summary user_count matches users table")
def g2(conn):
    s = scalar(conn, "SELECT CAST(value AS INTEGER) FROM ratings_summary WHERE key = \'user_count\'")
    n = scalar(conn, "SELECT COUNT(*) FROM users")
    return (s == n, f"summary {s} vs actual {n}")

@check("G3 summary rating_count is the known dataset constant")
def g3(conn):
    s = scalar(conn, "SELECT CAST(value AS INTEGER) FROM ratings_summary WHERE key = \'rating_count\'")
    return (s == EXPECTED["rating_count"], f"{s}")

@check("G4 generated_at parses as ISO-8601")
def g4(conn):
    v = scalar(conn, "SELECT value FROM ratings_summary WHERE key = \'generated_at\'")
    try:
        datetime.fromisoformat(v)
        return (True, v)
    except (TypeError, ValueError):
        return (False, repr(v))

#### Run the full report against the real artifact

In [13]:
ok = run(DB_PATH)
assert ok, "validation failed against the real artifact"

PASS  A1 file exists and >= 1 MB  [9.15 MB]
PASS  A2 opens read-only as SQLite
PASS  A3 all six tables exist
PASS  A4 all four indexes exist
PASS  A5 expected columns per table
PASS  B1 movies count  [9742]
PASS  B2 users count  [610]
PASS  B3 popular_movies count  [100]
PASS  B4 user_recommendations rows + full user coverage  [12200 rows, 610 users]
PASS  B5 movies with similarity rows (expected 5-gap)  [9737 (gap 5)]
PASS  B6 ratings_summary has 5 keys incl generated_at  [dataset_name, generated_at, movie_count, rating_count, user_count]
PASS  C1 user_recommendations.movie_id -> movies
PASS  C2 movie_similarity.movie_id -> movies
PASS  C3 movie_similarity.similar_movie_id -> movies
PASS  C4 popular_movies.movie_id -> movies
PASS  C5 user_recommendations.user_id -> users
PASS  D1 no self-similarity
PASS  D2 movie_similarity ranks contiguous 1..N per movie
PASS  D3 user_recommendations: exactly 20, ranks contiguous, per user
PASS  D4 popular_movies ranks are exactly 1..100  [min 1, max

#### Break it on purpose

A validator you\u2019ve never seen fail is itself unvalidated. Corrupt a *copy*, watch the right checks fail (expect **B4**, **D3**, **F1**), and confirm the real artifact still passes.

In [14]:
broken = "/tmp/movies_broken.db"
shutil.copy(DB_PATH, broken)

bconn = sqlite3.connect(broken)
bconn.execute('DELETE FROM user_recommendations WHERE user_id = 57 AND "rank" > 10')
bconn.commit()
bconn.close()

print("--- report against deliberately broken copy ---")
ok_broken = run(broken)
assert not ok_broken, "validator failed to catch a broken artifact"
print("\n--- real artifact, untouched (only ever opened read-only) ---")
assert run(DB_PATH)

--- report against deliberately broken copy ---


PASS  A1 file exists and >= 1 MB  [9.15 MB]
PASS  A2 opens read-only as SQLite
PASS  A3 all six tables exist
PASS  A4 all four indexes exist
PASS  A5 expected columns per table
PASS  B1 movies count  [9742]
PASS  B2 users count  [610]
PASS  B3 popular_movies count  [100]
FAIL  B4 user_recommendations rows + full user coverage  [12190 rows, 610 users]
PASS  B5 movies with similarity rows (expected 5-gap)  [9737 (gap 5)]
PASS  B6 ratings_summary has 5 keys incl generated_at  [dataset_name, generated_at, movie_count, rating_count, user_count]
PASS  C1 user_recommendations.movie_id -> movies
PASS  C2 movie_similarity.movie_id -> movies
PASS  C3 movie_similarity.similar_movie_id -> movies
PASS  C4 popular_movies.movie_id -> movies
PASS  C5 user_recommendations.user_id -> users
PASS  D1 no self-similarity
PASS  D2 movie_similarity ranks contiguous 1..N per movie
FAIL  D3 user_recommendations: exactly 20, ranks contiguous, per user  [1 bad groups]
PASS  D4 popular_movies ranks are exactly 1..

PASS  A1 file exists and >= 1 MB  [9.15 MB]
PASS  A2 opens read-only as SQLite
PASS  A3 all six tables exist
PASS  A4 all four indexes exist
PASS  A5 expected columns per table
PASS  B1 movies count  [9742]
PASS  B2 users count  [610]
PASS  B3 popular_movies count  [100]
PASS  B4 user_recommendations rows + full user coverage  [12200 rows, 610 users]
PASS  B5 movies with similarity rows (expected 5-gap)  [9737 (gap 5)]
PASS  B6 ratings_summary has 5 keys incl generated_at  [dataset_name, generated_at, movie_count, rating_count, user_count]
PASS  C1 user_recommendations.movie_id -> movies
PASS  C2 movie_similarity.movie_id -> movies
PASS  C3 movie_similarity.similar_movie_id -> movies
PASS  C4 popular_movies.movie_id -> movies
PASS  C5 user_recommendations.user_id -> users
PASS  D1 no self-similarity
PASS  D2 movie_similarity ranks contiguous 1..N per movie
PASS  D3 user_recommendations: exactly 20, ranks contiguous, per user
PASS  D4 popular_movies ranks are exactly 1..100  [min 1, max